In [ ]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [ ]:
# ------------------------------------------------------------
# connect DuckDB to the integrated PostgreSQL data source
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

In [ ]:
# ------------------------------------------------------------
# analytical-dataset construction context
# ------------------------------------------------------------

# the previous notebook selected the target and explanatory attributes
# this notebook turns that decision into a reproducible analytical dataset
# each row represents one eligible penguin observation
# the next steps define observation eligibility, then construct, validate and export the dataset

In [ ]:
# ------------------------------------------------------------
# guiding construction questions — analytical-dataset
# ------------------------------------------------------------

# IMPORTANT:
# these questions are not answered immediately
# the following cells collect the evidence needed to answer them

# CQ1. which observations are eligible for the analytical dataset?
# CQ2. how does the eligibility rule affect the dataset size and target distribution?
# CQ3. does the constructed dataset preserve the intended observation-unit and data-grain?

In [ ]:
# ------------------------------------------------------------
# construct the selected analytical source
# ------------------------------------------------------------

# retrieve the target and explanatory attributes selected in the previous analysis
# also recall the concepts of "observation-unit" and "data-grain"
#
# observation-unit:
# - one observation of one penguin
# - the same penguin may therefore contribute more than one observation
#
# data-grain:
# - defines the level of detail at which each row represents the observation-unit
# - here, each row must represent one specific penguin observed at one specific date_time
# - therefore, the intended data-grain is uniquely identified by (id_penguin, date_time)
#
# id_penguin and date_time were not selected as explanatory attributes
# they are temporarily retained to validate that the constructed data
# preserves the intended observation-unit and data-grain

db.sql("""
CREATE OR REPLACE TEMP VIEW penguins_selected AS
SELECT
    o.id_penguin,
    o.date_time,

    -- TODO: include the candidate explanatory attributes
    --       selected in the previous analysis
    ...,

    -- TODO: include the target attribute
    ...

FROM pg.public.observation AS o JOIN pg.public.penguin AS p
     ON o.id_penguin = p.id;
""")

In [ ]:
# ------------------------------------------------------------
# inspect missing values in the selected attributes
# ------------------------------------------------------------

# quantify missing values before defining the observation-eligibility rule
#
# recall:
# - COUNT(*) counts all rows
# - COUNT(attribute) counts only rows where attribute is not NULL
# - therefore, COUNT(*) - COUNT(attribute) gives the number of NULL values

db.sql("""
SELECT
    COUNT(*) AS n_observations,

    -- TODO: for each selected explanatory attribute and the target,
    --       count the number of missing values
    ...

FROM penguins_selected;
""").show()

# expected result:
# - 344 observations
# - year, island and species have no missing values
# - sex has 11 missing values
# - each morphological measurement has 2 missing values

In [ ]:
# ------------------------------------------------------------
# TODO
# conclude on CQ1 — observation eligibility
# ------------------------------------------------------------

# task:
# - use the missing-value counts obtained above
# - define which observations are eligible for the analytical dataset
# - consider the target and all selected explanatory attributes
# - for this analytical path, use a complete-case eligibility rule
# - id_penguin and date_time are retained only for validation and
#   do not determine observation eligibility

# conclusion for CQ1
# - an observation is eligible if:
#   ...

In [ ]:
# ------------------------------------------------------------
# construct the analytical dataset
# ------------------------------------------------------------

# apply the observation-eligibility rule defined above
#
# id_penguin and date_time are not explanatory attributes
# and are therefore excluded from the analytical dataset
# they remain available in penguins_selected for the later validation
# of the intended observation-unit and data-grain

db.sql("""
CREATE OR REPLACE TEMP VIEW analytical_dataset AS
SELECT
    island,
    year,
    sex,
    bill_length_mm,
    bill_depth_mm,
    flipper_length_mm,
    body_mass_g,
    species
FROM penguins_selected
WHERE
    -- TODO: implement the complete-case eligibility rule
    --       for all selected explanatory attributes and the target
    ...;
""")

In [ ]:
# ------------------------------------------------------------
# validate the analytical dataset
# ------------------------------------------------------------

# check the number of eligible observations
db.sql("""
SELECT
    COUNT(*) AS n_observations
FROM analytical_dataset;
""").show()

# inspect the target distribution among eligible observations
db.sql("""
SELECT
    species,
    COUNT(*) AS n_observations
FROM analytical_dataset
GROUP BY species
ORDER BY n_observations DESC;
""").show()

In [ ]:
# ------------------------------------------------------------
# validate observation-unit and data-grain
# ------------------------------------------------------------

# verify that each eligible row represents one observation-unit
# and that the intended data-grain is preserved:
# - one specific penguin observed at one specific date_time
#
# therefore, (id_penguin, date_time) must uniquely identify
# every eligible observation

db.sql("""
SELECT
    COUNT(*) AS n_observations,

    -- TODO: count the distinct observation-units identified by
    --       (id_penguin, date_time)
    ... AS n_observation_units,

    -- TODO: compute how many observations violate the intended data-grain
    --       by repeating the same (id_penguin, date_time)
    ... AS n_duplicate_observations

FROM penguins_selected
WHERE

    -- TODO: apply the same observation-eligibility rule
    --       used to construct analytical_dataset
    ...;
""").show()

# expected result:
# n_observations = 333
# n_observation_units = 333
# n_duplicate_observations = 0

In [ ]:
# ------------------------------------------------------------
# compare target distribution before and after observation eligibility
# ------------------------------------------------------------

# use two Common Table Expressions (CTEs) to summarize the target
# distribution before and after applying the eligibility rule
#
# a CTE, defined with WITH, creates a named intermediate result
# that can be used within the same SQL statement

db.sql("""
WITH

before_eligibility AS (
SELECT
    species,
    COUNT(*) AS n_before,

    -- TODO: compute the percentage of each species before eligibility
    --       reuse the window-function pattern used previously
    ... AS pct_before

FROM penguins_selected
GROUP BY species
),

after_eligibility AS (
SELECT
    species,
    COUNT(*) AS n_after,

    -- TODO: compute the percentage of each species after eligibility
    ... AS pct_after

FROM analytical_dataset
GROUP BY species
)

SELECT
    b.species,
    b.n_before,
    a.n_after,
    ROUND(b.pct_before, 2) AS pct_before,
    ROUND(a.pct_after, 2) AS pct_after,

    -- TODO: compute the change in percentage points
    ... AS percentage_point_difference

FROM before_eligibility AS b JOIN after_eligibility AS a
     ON b.species = a.species
ORDER BY b.species;
""").show()

# expected result:
# 3 rows, one for each species
# the largest absolute change in class percentage is less than 1 percentage point

In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for CQ2 and CQ3
# ------------------------------------------------------------

# task:
# - use the validation results obtained above
# - answer the evidence questions and conclude on CQ2 and CQ3

# E1. the individual missing-value counts sum to more than the number of excluded observations
# - what does this indicate about missing values across attributes?
# A1:

# E2. by how many observations is the dataset reduced,
#     and what is the largest change in class percentage?
# A2:

# conclusion for CQ2
# A:

# E3. what does n_duplicate_observations indicate about
#     the intended observation-unit and data-grain?
# A3:

# E4. why are id_penguin and date_time useful for validation
#     but excluded from the analytical dataset?
# A4:

# conclusion for CQ3
# A:


#
# reminder (possibly useful for E4):
# observation-unit = what one record represents conceptually
# data-grain = the level of detail at which that observation-unit is represented
#
# in this MoP:
# observation-unit = one observation of one penguin
# data-grain = one specific penguin observed at one specific date_time
#              identified by (id_penguin, date_time)

In [ ]:
# ------------------------------------------------------------
# materialize the analytical dataset
# ------------------------------------------------------------

from pathlib import Path

output_file = Path("../03_output/penguins_analytical_dataset.csv").resolve()
output_file.parent.mkdir(parents=True, exist_ok=True)

db.sql(f"""
COPY analytical_dataset
TO '{ output_file.as_posix() }'
(FORMAT CSV, HEADER);
""")

print(output_file)

# Second visual checkpoint with Orange Data Mining (DM)

Before starting this checkpoint, make sure that the artefacts required from MoP_02
have been copied into the current MoP workspace.

In particular, you need:

- `01_input/_mLDm_00_penguins_data_collected.csv`
- `02_code/penguins_visual_check.ows`

The Orange workflow was created in MoP_02 and will now be extended rather than
recreated from scratch.

## 1. Reopen the workflow created in MoP_02

Open:

`02_code/penguins_visual_check.ows`

Keep the existing `File` widget connected to:

`01_input/_mLDm_00_penguins_data_collected.csv`

This branch corresponds to the initial collected source explored in notebook 02.

## 2. Add the analytical dataset

Add a second `File` widget and load, from the current MoP:

`03_output/penguins_analytical_dataset.csv`

Connect this new `File` widget to a new `Select Columns` widget.

For the analytical dataset, verify that:

- `species` is the **Target**
- the remaining attributes are **Features**
- attributes excluded during the previous analysis are no longer present

## 3. Compare both data representations

For each branch, connect `Select Columns` to:

- `Data Table`
- `Distributions`
- `Scatter Plot`

Use the two branches to compare:

- the attributes available in each dataset
- the roles assigned to those attributes
- the number and nature of the observations
- the visual distributions and relationships between selected attributes

Focus particularly on how the analytical dataset differs from the
initial collected source as a consequence of the decisions made in
notebooks 03 and 04.

Do not train or compare predictive models yet.

The purpose of this checkpoint is to compare the **collected source**
with the **materialized analytical dataset** before moving to the modelling stage.

Save the extended workflow again as:

`02_code/penguins_visual_check.ows`